In [ ]:
"""
Random Forest + SHAP Analysis Demo
====================================
Demonstrates a full Random Forest classification workflow with SHAP
explanations, diagnostic plots, and model evaluation.

Uses the Breast Cancer Wisconsin dataset (built into sklearn).
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec  # (not used, can be removed)
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)
from sklearn.inspection import permutation_importance
import shap
import warnings
warnings.filterwarnings("ignore")

# Set for Output Directory (adjust this path as needed)

from pathlib import Path

OUTPUT_DIR = Path("/Users/rudolphsurovcik/Library/CloudStorage/OneDrive-Personal/Python Folder")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Then every save call becomes:
#plt.savefig(OUTPUT_DIR / "02_permutation_importance.png", dpi=150, bbox_inches="tight")
#plt.savefig(OUTPUT_DIR / "03_shap_beeswarm.png", dpi=150, bbox_inches="tight")

# ── Style ────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f8f9fa",
    "axes.edgecolor": "#cccccc",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.family": "sans-serif",
    "font.size": 11,
})
COLORS = ["#2563eb", "#e11d48", "#16a34a", "#f59e0b", "#8b5cf6"]

# ═══════════════════════════════════════════════════════════════════════════════
# 1. DATA LOADING & EXPLORATION
# ═══════════════════════════════════════════════════════════════════════════════
print("=" * 72)
print("RANDOM FOREST + SHAP ANALYSIS")
print("=" * 72)

data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df["target"] = data.target  # 0 = malignant, 1 = benign

print(f"\nDataset shape        : {df.shape}")
print(f"Features             : {len(data.feature_names)}")
print(f"Classes              : {dict(zip(data.target_names, np.bincount(data.target)))}")
print(f"\nFirst 5 rows:\n{df.head()}\n")

X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Train size: {len(X_train)}  |  Test size: {len(X_test)}")

# ═══════════════════════════════════════════════════════════════════════════════
# 2. RANDOM FOREST MODEL
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 72)
print("TRAINING RANDOM FOREST")
print("─" * 72)

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

train_acc = rf.score(X_train, y_train)
test_acc = rf.score(X_test, y_test)
print(f"\nTrain accuracy : {train_acc:.4f}")
print(f"Test accuracy  : {test_acc:.4f}")

cv_scores = cross_val_score(rf, X, y, cv=5, scoring="accuracy")
print(f"5-Fold CV      : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# ═══════════════════════════════════════════════════════════════════════════════
# 3. CLASSIFICATION DIAGNOSTICS
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 72)
print("CLASSIFICATION REPORT")
print("─" * 72)

y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=data.target_names))

# ── Figure 1: Model Diagnostics (2×2) ───────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle("Random Forest — Model Diagnostics", fontsize=16, fontweight="bold", y=0.98)

# (a) Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
im = axes[0, 0].imshow(cm, cmap="Blues", aspect="auto")
axes[0, 0].set_xticks([0, 1])
axes[0, 0].set_yticks([0, 1])
axes[0, 0].set_xticklabels(data.target_names, fontsize=10)
axes[0, 0].set_yticklabels(data.target_names, fontsize=10)
axes[0, 0].set_xlabel("Predicted")
axes[0, 0].set_ylabel("Actual")
axes[0, 0].set_title("(a) Confusion Matrix")
for i in range(2):
    for j in range(2):
        axes[0, 0].text(j, i, str(cm[i, j]), ha="center", va="center",
                        fontsize=20, fontweight="bold",
                        color="white" if cm[i, j] > cm.max() / 2 else "black")

# (b) ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)
axes[0, 1].plot(fpr, tpr, color=COLORS[0], lw=2.5, label=f"AUC = {roc_auc:.3f}")
axes[0, 1].plot([0, 1], [0, 1], "k--", lw=1, alpha=0.4)
axes[0, 1].fill_between(fpr, tpr, alpha=0.08, color=COLORS[0])
axes[0, 1].set(xlabel="False Positive Rate", ylabel="True Positive Rate",
               title="(b) ROC Curve", xlim=[-0.01, 1.01], ylim=[-0.01, 1.05])
axes[0, 1].legend(loc="lower right", fontsize=12)

# (c) Precision-Recall Curve
prec, rec, _ = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)
axes[1, 0].plot(rec, prec, color=COLORS[1], lw=2.5, label=f"AP = {ap:.3f}")
axes[1, 0].fill_between(rec, prec, alpha=0.08, color=COLORS[1])
axes[1, 0].set(xlabel="Recall", ylabel="Precision",
               title="(c) Precision-Recall Curve", xlim=[-0.01, 1.01], ylim=[-0.01, 1.05])
axes[1, 0].legend(loc="lower left", fontsize=12)

# (d) Top 15 Feature Importances (Gini)
importances = rf.feature_importances_
sorted_idx = np.argsort(importances)[-15:]
axes[1, 1].barh(range(15), importances[sorted_idx], color=COLORS[0], edgecolor="white")
axes[1, 1].set_yticks(range(15))
axes[1, 1].set_yticklabels(X.columns[sorted_idx], fontsize=9)
axes[1, 1].set_xlabel("Gini Importance")
axes[1, 1].set_title("(d) Top 15 Feature Importances")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# ═══════════════════════════════════════════════════════════════════════════════
# 4. PERMUTATION IMPORTANCE (comparison to Gini)
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 72)
print("PERMUTATION IMPORTANCE")
print("─" * 72)

perm = permutation_importance(rf, X_test, y_test, n_repeats=15, random_state=42, n_jobs=-1)
perm_sorted = np.argsort(perm.importances_mean)[-15:]

fig, ax = plt.subplots(figsize=(10, 7))
ax.boxplot(
    perm.importances[perm_sorted].T,
    vert=False,
    patch_artist=True,
    boxprops=dict(facecolor=COLORS[4], alpha=0.6),
    medianprops=dict(color="black"),
)
ax.set_yticklabels(X.columns[perm_sorted], fontsize=9)
ax.set_xlabel("Decrease in Accuracy")
ax.set_title("Permutation Importance (Test Set, 15 repeats)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "02_permutation_importance.png", dpi=150, bbox_inches="tight")
plt.tight_layout()
plt.show()
# 5. SHAP ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 72)
print("SHAP ANALYSIS")
print("─" * 72)

explainer = shap.TreeExplainer(rf)
shap_values = explainer(X_test)

# For binary classification, shap_values has shape (n_samples, n_features, 2)
# We take the SHAP values for class 1 (benign)
sv_class1 = shap_values[..., 1]

# ── Figure 3: SHAP Summary (Beeswarm) ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 8))
shap.plots.beeswarm(sv_class1, max_display=20, show=False)
plt.title("SHAP Beeswarm Plot — Feature Impact on P(Benign)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03_shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.close()
plt.title("SHAP Beeswarm Plot — Feature Impact on P(Benign)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# SHAP bar plot for mean(|SHAP|) global importance
mean_shap = np.abs(sv_class1.values).mean(axis=0)
feat_order = np.argsort(mean_shap)[-20:][::-1]
plt.figure(figsize=(10, 7))
plt.barh(range(len(feat_order)), mean_shap[feat_order], color=COLORS[0])
plt.yticks(range(len(feat_order)), X.columns[feat_order], fontsize=9)
plt.xlabel("Mean |SHAP Value|")
plt.title("Mean |SHAP Value| — Global Feature Importance", fontsize=14, fontweight="bold")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "04_shap_bar.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved → 04_shap_bar.png")

# Find a malignant and benign sample index in the test set
mal_idx = np.where(y_test.values == 0)[0][0]
ben_idx = np.where(y_test.values == 1)[0][0]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

plt.sca(axes[0])
shap.plots.waterfall(sv_class1[mal_idx], max_display=12, show=False)
axes[0].set_title(f"Waterfall — Malignant Sample (idx {mal_idx})", fontsize=12, fontweight="bold")

plt.sca(axes[1])
shap.plots.waterfall(sv_class1[ben_idx], max_display=12, show=False)
axes[1].set_title(f"Waterfall — Benign Sample (idx {ben_idx})", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "05_shap_waterfall.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved → 05_shap_waterfall.png")

# ── Figure 6: SHAP Dependence Plots (top 4 features) ────────────────────────
top4 = np.argsort(mean_shap)[-4:][::-1]
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for i, feat_idx in enumerate(top4):
    feat_name = X.columns[feat_idx]
fig, ax = plt.subplots(figsize=(14, 8))
shap.plots.heatmap(sv_class1, max_display=15, show=False)
plt.title("SHAP Heatmap — Per-Sample Feature Contributions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "07_shap_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
plt.show()

# ═══════════════════════════════════════════════════════════════════════════════
# 6. SUMMARY TABLE
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 72)
print("SUMMARY")
print("=" * 72)

summary = pd.DataFrame({
    "Feature": X.columns,
    "Gini Importance": rf.feature_importances_,
    "Permutation Imp.": perm.importances_mean,
    "Mean |SHAP|": mean_shap,
}).sort_values("Mean |SHAP|", ascending=False).head(15).reset_index(drop=True)

summary.index += 1
print(f"\nTop 15 features by Mean |SHAP Value|:\n")
print(summary.to_string())

print("\n\nOutput files:")
for f in [
    "01_rf_diagnostics.png",
    "02_permutation_importance.png",
    "03_shap_beeswarm.png",
    "04_shap_bar.png",
    "05_shap_waterfall.png",
    "06_shap_dependence.png",
    "07_shap_heatmap.png",
]:
    print(f"  • {f}")
print("\nDone.")
print("\n\nAll plots displayed inline above.")
print("\nDone.")

RANDOM FOREST + SHAP ANALYSIS

Dataset shape        : (569, 31)
Features             : 30
Classes              : {np.str_('malignant'): np.int64(212), np.str_('benign'): np.int64(357)}

First 5 rows:
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414        